In [1]:
import pandas as pd

# 读取 Excel 文件
FARE = pd.read_excel("../Data/BUS/SECTION_FARE.xlsx", engine="openpyxl")  

# 查看前几行数据
print(FARE.head())



   ROUTE_ID  ROUTE_SEQ  ON_SEQ  OFF_SEQ  PRICE LAST_UPDATE_DATE
0      1001          1       1        2    6.7       2025-01-02
1      1001          1       1        3    6.7       2025-01-02
2      1001          1       1        4    6.7       2025-01-02
3      1001          1       1        5    6.7       2025-01-02
4      1001          1       1        6    6.7       2025-01-02


In [2]:
ROUTE = pd.read_excel("..\Data\BUS\ROUTE.xlsx", engine="openpyxl")  

# 查看前几行数据
print(ROUTE.head())


   ROUTE_ID COMPANY_CODE  DISTRICT ROUTE_NAMEC ROUTE_NAMES ROUTE_NAMEE  \
0      1001          KMB       NaN           1           1           1   
1      1002          KMB       NaN          10          10          10   
2      1006      KMB+CTB       NaN        102P        102P        102P   
3      1008      KMB+CTB       NaN         103         103         103   
4      1010      KMB+CTB       NaN         106         106         106   

   ROUTE_TYPE SERVICE_MODE  SPECIAL_TYPE  JOURNEY_TIME  ... LOC_START_NAMES  \
0           1            R             0            40  ...             竹园邨   
1           1            R             0            55  ...              彩云   
2           1            T             1            64  ...             筲箕湾   
3           1            R             0            59  ...             竹园邨   
4           1            R             0            82  ...             黄大仙   

    LOC_START_NAMEE LOC_END_NAMEC LOC_END_NAMES                LOC_END_NAMEE  \


In [3]:
RSTOP = pd.read_excel("..\Data\BUS\ROUTE_STOP.xlsx", engine="openpyxl")  

print(RSTOP.head())


   ROUTE_ID  ROUTE_SEQ  STOP_SEQ  STOP_ID  STOP_PICK_DROP STOP_NAMEC  \
0      1001          1         1     4001               2      竹園邨總站   
1      1001          1         2     4002               3       天虹小學   
2      1001          1         3     4003               3     馬仔坑遊樂場   
3      1001          1         4     4004               3       摩士公園   
4      1001          1         5     4005               3    摩士公園體育館   

  STOP_NAMES                      STOP_NAMEE LAST_UPDATE_DATE  
0      竹园邨总站   CHUK YUEN ESTATE BUS TERMINUS       2023-02-21  
1       天虹小学          RAINBOW PRIMARY SCHOOL       2023-02-21  
2     马仔坑游乐场  MA CHAI HANG RECREATION GROUND       2023-02-21  
3       摩士公园                      MORSE PARK       2023-02-21  
4    摩士公园体育馆        MORSE PARK SPORTS CENTRE       2023-02-21  


In [4]:
STOP = pd.read_excel("..\Data\BUS\STOP_XY.xlsx", engine="openpyxl")  

print(STOP.head())


   STOP_ID  STOP_TYPE       X       Y LAST_UPDATE_DATE  STOP_CODE
0        2          1  843622  813951       2024-02-02        NaN
1        3          1  843333  814109       2023-03-04        NaN
2        4          1  842947  814013       2023-03-04        NaN
3        5          1  842646  813814       2023-03-04        NaN
4        6          1  842419  813736       2023-03-04        NaN


In [5]:
ROUTE.columns

Index(['ROUTE_ID', 'COMPANY_CODE', 'DISTRICT', 'ROUTE_NAMEC', 'ROUTE_NAMES',
       'ROUTE_NAMEE', 'ROUTE_TYPE', 'SERVICE_MODE', 'SPECIAL_TYPE',
       'JOURNEY_TIME', 'LOC_START_NAMEC', 'LOC_START_NAMES', 'LOC_START_NAMEE',
       'LOC_END_NAMEC', 'LOC_END_NAMES', 'LOC_END_NAMEE', 'HYPERLINK_C',
       'HYPERLINK_S', 'HYPERLINK_E', 'FULL_FARE', 'LAST_UPDATE_DATE'],
      dtype='object')

In [6]:
import pandas as pd


fare_with_route = FARE.merge(ROUTE[['ROUTE_ID', 'ROUTE_NAMES','COMPANY_CODE','SERVICE_MODE','SPECIAL_TYPE','JOURNEY_TIME']], on='ROUTE_ID', how='left')
#因为假设只是区域内的交通，大部分巴士都是跨区的，同时Journey time只有整条巴士路线的数据，因此暂时舍弃Journey time

# 2. 关联 RSTOP 表，获取 ON_SEQ_NAMES 和 ON_SEQ_STOP_ID
fare_with_on_seq = fare_with_route.merge(
    RSTOP[['ROUTE_ID', 'ROUTE_SEQ', 'STOP_SEQ', 'STOP_NAMES', 'STOP_ID']],
    left_on=['ROUTE_ID', 'ROUTE_SEQ', 'ON_SEQ'],
    right_on=['ROUTE_ID', 'ROUTE_SEQ', 'STOP_SEQ'],
    how='left'
).rename(columns={'STOP_NAMEC': 'ON_SEQ_NAMES', 'STOP_ID': 'ON_SEQ_STOP_ID'})

# 3. 关联 RSTOP 表，获取 OFF_SEQ_NAMES 和 OFF_SEQ_STOP_ID
fare_with_off_seq = fare_with_on_seq.merge(
    RSTOP[['ROUTE_ID', 'ROUTE_SEQ', 'STOP_SEQ', 'STOP_NAMES', 'STOP_ID']],
    left_on=['ROUTE_ID', 'ROUTE_SEQ', 'OFF_SEQ'],
    right_on=['ROUTE_ID', 'ROUTE_SEQ', 'STOP_SEQ'],
    how='left'
).rename(columns={'STOP_NAMEC': 'OFF_SEQ_NAMES', 'STOP_ID': 'OFF_SEQ_STOP_ID'})


fare_with_on_XY=fare_with_off_seq.merge(
    STOP[['STOP_ID','X','Y']],
    left_on=['ON_SEQ_STOP_ID'],
    right_on=['STOP_ID'],
    how='left'
).rename(columns={'X':'ON_SEQ_STOP_X','Y':'ON_SEQ_STOP_Y'})

fare_with_off_XY=fare_with_on_XY.merge(
    STOP[['STOP_ID','X','Y']],
    left_on=['OFF_SEQ_STOP_ID'],
    right_on=['STOP_ID'],
    how='left'
).rename(columns={'X':'OFF_SEQ_STOP_X','Y':'OFF_SEQ_STOP_Y'})

# 5. 查看结果
print(fare_with_off_XY.head())

# 6. 将结果存入新的 DataFrame
fare_with_off_XY




   ROUTE_ID  ROUTE_SEQ  ON_SEQ  OFF_SEQ  PRICE LAST_UPDATE_DATE ROUTE_NAMES  \
0      1001          1       1        2    6.7       2025-01-02           1   
1      1001          1       1        3    6.7       2025-01-02           1   
2      1001          1       1        4    6.7       2025-01-02           1   
3      1001          1       1        5    6.7       2025-01-02           1   
4      1001          1       1        6    6.7       2025-01-02           1   

  COMPANY_CODE SERVICE_MODE  SPECIAL_TYPE  ...  ON_SEQ_STOP_ID  STOP_SEQ_y  \
0          KMB            R             0  ...            4001           2   
1          KMB            R             0  ...            4001           3   
2          KMB            R             0  ...            4001           4   
3          KMB            R             0  ...            4001           5   
4          KMB            R             0  ...            4001           6   

  STOP_NAMES_y  OFF_SEQ_STOP_ID  STOP_ID_x ON_SEQ_STOP_X

,ROUTE_ID,ROUTE_SEQ,ON_SEQ,OFF_SEQ,PRICE,LAST_UPDATE_DATE,ROUTE_NAMES,COMPANY_CODE,SERVICE_MODE,SPECIAL_TYPE,...,ON_SEQ_STOP_ID,STOP_SEQ_y,STOP_NAMES_y,OFF_SEQ_STOP_ID,STOP_ID_x,ON_SEQ_STOP_X,ON_SEQ_STOP_Y,STOP_ID_y,OFF_SEQ_STOP_X,OFF_SEQ_STOP_Y
0,1001,1,1,2,6.7,2025-01-02,1,KMB,R,0,...,4001,2,天虹小学,4002,4001,837872,822918,4002,837621,822886
1,1001,1,1,3,6.7,2025-01-02,1,KMB,R,0,...,4001,3,马仔坑游乐场,4003,4001,837872,822918,4003,837391,822793
2,1001,1,1,4,6.7,2025-01-02,1,KMB,R,0,...,4001,4,摩士公园,4004,4001,837872,822918,4004,837557,822294
3,1001,1,1,5,6.7,2025-01-02,1,KMB,R,0,...,4001,5,摩士公园体育馆,4005,4001,837872,822918,4005,837610,822082
4,1001,1,1,6,6.7,2025-01-02,1,KMB,R,0,...,4001,6,伟东楼,4006,4001,837872,822918,4006,837768,821807
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755821,1000575,1,7,14,5.1,2025-01-17,272A,KMB,R,0,...,13002,14,科学园(第一期),13090,13002,838993,832234,13090,839844,831661
755822,1000575,1,7,15,5.1,2025-01-17,272A,KMB,R,0,...,13002,15,香港生物科技研究院,12401,13002,838993,832234,12401,840110,831456
755823,1000575,1,7,16,5.1,2025-01-17,272A,KMB,R,0,...,13002,16,水上活动中心,12402,13002,838993,832234,12402,840119,831046
755824,1000575,1,7,17,5.1,2025-01-17,272A,KMB,R,0,...,13002,17,马料水公众码头,10000018,13002,838993,832234,10000018,840085,830744


In [7]:
STOP_DISTRICT=pd.read_excel("../Data/BUS/STOP_DISTRICT.xlsx",engine="openpyxl")
STOP_DISTRICT.head()

,地區號碼,District,地區,Administrative District Boundary of Hong Kong,OBJECTID,STOP_ID,LAST_UPDATE_DATE
0,A,Central & Western,中西區,Administrative District Boundary of Hong Kong,19,21,2023-03-04
1,A,Central & Western,中西區,Administrative District Boundary of Hong Kong,20,22,2023-03-04
2,A,Central & Western,中西區,Administrative District Boundary of Hong Kong,21,23,2023-03-04
3,A,Central & Western,中西區,Administrative District Boundary of Hong Kong,22,24,2023-03-04
4,A,Central & Western,中西區,Administrative District Boundary of Hong Kong,23,26,2023-03-04


In [8]:
fare_with_on_district=fare_with_off_XY.merge(
    STOP_DISTRICT[['地區','STOP_ID']],
    left_on=['ON_SEQ_STOP_ID'],
    right_on=['STOP_ID'],
    how='left'
).rename(columns={'地區':'ON_DISTRICT','STOP_ID':'Deplicate_STOP_ID'})
fare_with_off_district=fare_with_on_district.merge(
    STOP_DISTRICT[['地區','STOP_ID']],
    left_on=['OFF_SEQ_STOP_ID'],
    right_on=['STOP_ID'],
    how='left'
).rename(columns={'地區':'OFF_DISTRICT','STOP_ID_x':'DUPLICATE_STOP_ID_2'})
fare_with_off_district

,ROUTE_ID,ROUTE_SEQ,ON_SEQ,OFF_SEQ,PRICE,LAST_UPDATE_DATE,ROUTE_NAMES,COMPANY_CODE,SERVICE_MODE,SPECIAL_TYPE,...,DUPLICATE_STOP_ID_2,ON_SEQ_STOP_X,ON_SEQ_STOP_Y,STOP_ID_y,OFF_SEQ_STOP_X,OFF_SEQ_STOP_Y,ON_DISTRICT,Deplicate_STOP_ID,OFF_DISTRICT,STOP_ID
0,1001,1,1,2,6.7,2025-01-02,1,KMB,R,0,...,4001,837872,822918,4002,837621,822886,黃大仙,4001.0,黃大仙,4002.0
1,1001,1,1,3,6.7,2025-01-02,1,KMB,R,0,...,4001,837872,822918,4003,837391,822793,黃大仙,4001.0,黃大仙,4003.0
2,1001,1,1,4,6.7,2025-01-02,1,KMB,R,0,...,4001,837872,822918,4004,837557,822294,黃大仙,4001.0,黃大仙,4004.0
3,1001,1,1,5,6.7,2025-01-02,1,KMB,R,0,...,4001,837872,822918,4005,837610,822082,黃大仙,4001.0,黃大仙,4005.0
4,1001,1,1,6,6.7,2025-01-02,1,KMB,R,0,...,4001,837872,822918,4006,837768,821807,黃大仙,4001.0,黃大仙,4006.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755821,1000575,1,7,14,5.1,2025-01-17,272A,KMB,R,0,...,13002,838993,832234,13090,839844,831661,大埔,13002.0,沙田,13090.0
755822,1000575,1,7,15,5.1,2025-01-17,272A,KMB,R,0,...,13002,838993,832234,12401,840110,831456,大埔,13002.0,沙田,12401.0
755823,1000575,1,7,16,5.1,2025-01-17,272A,KMB,R,0,...,13002,838993,832234,12402,840119,831046,大埔,13002.0,沙田,12402.0
755824,1000575,1,7,17,5.1,2025-01-17,272A,KMB,R,0,...,13002,838993,832234,10000018,840085,830744,大埔,13002.0,沙田,10000018.0


In [9]:

final_fare_df=fare_with_off_district
final_fare_df=final_fare_df.drop(columns=['Deplicate_STOP_ID','DUPLICATE_STOP_ID_2','STOP_ID_y','STOP_ID'])
final_fare_df

,ROUTE_ID,ROUTE_SEQ,ON_SEQ,OFF_SEQ,PRICE,LAST_UPDATE_DATE,ROUTE_NAMES,COMPANY_CODE,SERVICE_MODE,SPECIAL_TYPE,...,ON_SEQ_STOP_ID,STOP_SEQ_y,STOP_NAMES_y,OFF_SEQ_STOP_ID,ON_SEQ_STOP_X,ON_SEQ_STOP_Y,OFF_SEQ_STOP_X,OFF_SEQ_STOP_Y,ON_DISTRICT,OFF_DISTRICT
0,1001,1,1,2,6.7,2025-01-02,1,KMB,R,0,...,4001,2,天虹小学,4002,837872,822918,837621,822886,黃大仙,黃大仙
1,1001,1,1,3,6.7,2025-01-02,1,KMB,R,0,...,4001,3,马仔坑游乐场,4003,837872,822918,837391,822793,黃大仙,黃大仙
2,1001,1,1,4,6.7,2025-01-02,1,KMB,R,0,...,4001,4,摩士公园,4004,837872,822918,837557,822294,黃大仙,黃大仙
3,1001,1,1,5,6.7,2025-01-02,1,KMB,R,0,...,4001,5,摩士公园体育馆,4005,837872,822918,837610,822082,黃大仙,黃大仙
4,1001,1,1,6,6.7,2025-01-02,1,KMB,R,0,...,4001,6,伟东楼,4006,837872,822918,837768,821807,黃大仙,黃大仙
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755821,1000575,1,7,14,5.1,2025-01-17,272A,KMB,R,0,...,13002,14,科学园(第一期),13090,838993,832234,839844,831661,大埔,沙田
755822,1000575,1,7,15,5.1,2025-01-17,272A,KMB,R,0,...,13002,15,香港生物科技研究院,12401,838993,832234,840110,831456,大埔,沙田
755823,1000575,1,7,16,5.1,2025-01-17,272A,KMB,R,0,...,13002,16,水上活动中心,12402,838993,832234,840119,831046,大埔,沙田
755824,1000575,1,7,17,5.1,2025-01-17,272A,KMB,R,0,...,13002,17,马料水公众码头,10000018,838993,832234,840085,830744,大埔,沙田


In [10]:

final_fare_df.to_excel('../Data/BUS/Converted_Fare.xlsx', index=False, engine='openpyxl')